# Notebook 01: Setup and Multi-Dataset Pipeline

This notebook sets up the project environment, handles data loading, processes both **IEEE-CIS Fraud Detection** and **PaySim Mobile Money** datasets, and generates synthetic labels for multi-task utilization.

In [ ]:
# Setup cell with Colab auto-detection
import os, sys
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Finai-research'
    if not os.path.exists(PROJECT_ROOT):
        !git clone https://github.com/outlieralpha/Finai-research.git {PROJECT_ROOT}
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    !pip install -q torch xgboost lightgbm catboost scikit-learn pandas numpy pyarrow matplotlib seaborn tqdm
else:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import config
paths = config.setup_environment()
print('Environment initialized. Path:', paths['project_root'])

## 1. Process PaySim Pre-training Dataset (6.3M transactions)

In [ ]:
from data.make_paysim_dataset import load_or_create_paysim, process_paysim_features, create_paysim_splits

raw_dir = paths['data_raw']
processed_dir = paths['data_processed']

df_paysim = load_or_create_paysim(raw_dir)
df_paysim_proc = process_paysim_features(df_paysim)
create_paysim_splits(df_paysim_proc, processed_dir)
print('PaySim pipeline complete.')

## 2. Process IEEE-CIS Downstream Dataset & Generate Synthetic Labels

In [ ]:
from data.make_dataset import load_data, process_features, create_splits
from synthetic_labels import attach_all_synthetic_labels
import pandas as pd

try:
    df_ieee = load_data(raw_dir)
    df_ieee_proc = process_features(df_ieee)
    create_splits(df_ieee_proc, processed_dir)
    
    # Generate synthetic labels
    synthetic_meta = attach_all_synthetic_labels(df_ieee_proc, client_col='ClientID', amount_col='TransactionAmt', time_col='TransactionDT')
    synthetic_meta.to_parquet(processed_dir / 'ieee_synthetic_labels.parquet')
    print('IEEE-CIS pipeline & synthetic label generation complete.')
except Exception as e:
    print('Note: Raw IEEE-CIS CSVs not in data/raw. Place train_transaction.csv & train_identity.csv in data/raw to run full IEEE-CIS pipeline.')